In [2]:
!pip install transformers datasets torch pandas accelerate scikit-learn

In [3]:
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
import time
import json

## Setup and Data Preparation

In [4]:
from google.colab import files
uploaded = files.upload()

Saving symptom-disease-dataset.zip to symptom-disease-dataset.zip


In [5]:
!unzip symptom-disease-dataset.zip -d unzipped_data/

Archive:  symptom-disease-dataset.zip
   creating: unzipped_data/symptom-disease-dataset/
   creating: unzipped_data/symptom-disease-dataset/.git/
  inflating: unzipped_data/symptom-disease-dataset/.git/config  
  inflating: unzipped_data/symptom-disease-dataset/.git/description  
  inflating: unzipped_data/symptom-disease-dataset/.git/FETCH_HEAD  
  inflating: unzipped_data/symptom-disease-dataset/.git/HEAD  
   creating: unzipped_data/symptom-disease-dataset/.git/hooks/
  inflating: unzipped_data/symptom-disease-dataset/.git/hooks/applypatch-msg.sample  
  inflating: unzipped_data/symptom-disease-dataset/.git/hooks/commit-msg.sample  
  inflating: unzipped_data/symptom-disease-dataset/.git/hooks/fsmonitor-watchman.sample  
  inflating: unzipped_data/symptom-disease-dataset/.git/hooks/post-update.sample  
  inflating: unzipped_data/symptom-disease-dataset/.git/hooks/pre-applypatch.sample  
  inflating: unzipped_data/symptom-disease-dataset/.git/hooks/pre-commit.sample  
  inflating: u

In [6]:
train_df = pd.read_csv("unzipped_data/symptom-disease-dataset/symptom-disease-train-dataset.csv")
test_df = pd.read_csv("unzipped_data/symptom-disease-dataset/symptom-disease-test-dataset.csv")

In [7]:
train_df = train_df.rename(columns={train_df.columns[0]: "text", train_df.columns[1]: "label"})
test_df = test_df.rename(columns={test_df.columns[0]: "text", test_df.columns[1]: "label"})

In [8]:
num_classes = max(train_df['label'].max(), test_df['label'].max()) + 1
num_classes

1082

In [9]:
raw_datasets = DatasetDict({
    "train": Dataset.from_pandas(train_df),
    "test": Dataset.from_pandas(test_df)
})

## Tokenization

In [13]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [14]:
def tokenize(examples):
  return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

In [15]:
tokenized_datasets = raw_datasets.map(tokenize, batched=True)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/5634 [00:00<?, ? examples/s]

Map:   0%|          | 0/1409 [00:00<?, ? examples/s]

## Initialize model and train

In [16]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_classes
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [19]:
training_args = TrainingArguments(
    output_dir="./wellnest_model_checkpoints",
    num_train_epochs=10,
    per_device_train_batch_size=16,
    # evaluation_strategy="epoch",
    # save_strategy="epoch",
    learning_rate=2e-5,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    # tokenizer=tokenizer,
    data_collator=data_collator,
)

In [20]:
trainer.train()

Step,Training Loss
500,3.815369
1000,1.277795
1500,1.044465
2000,0.993333
2500,0.933518
3000,0.894628
3500,0.865609


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3530, training_loss=1.40078827098814, metrics={'train_runtime': 675.267, 'train_samples_per_second': 83.434, 'train_steps_per_second': 5.228, 'total_flos': 1901739179612160.0, 'train_loss': 1.40078827098814, 'epoch': 10.0})

## Quantization to 8-bits

In [21]:
model.to("cpu")

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


In [25]:
quantized_model = torch.quantization.quantize_dynamic(
    model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

/tmp/ipykernel_550/2303141631.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [26]:
torch.save(quantized_model.state_dict(), "quantized_disease_model.pt")
tokenizer.save_pretrained("./quantized_tokenizer")

('./quantized_tokenizer/tokenizer_config.json',
 './quantized_tokenizer/tokenizer.json')

## TTS testing

In [27]:
!pip install soundfile librosa datasets

In [28]:
import torch
from transformers import pipeline, SpeechT5Processor, SpeechT5ForTextToSpeech, SpeechT5HifiGan
from datasets import load_dataset
from IPython.display import Audio, display

In [54]:
stt_pipeline = pipeline("automatic-speech-recognition", model="openai/whisper-tiny", chunk_length_s=30)

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


In [55]:
tts_processor = SpeechT5Processor.from_pretrained("microsoft/speecht5_tts")
tts_model = SpeechT5ForTextToSpeech.from_pretrained("microsoft/speecht5_tts")
vocoder = SpeechT5HifiGan.from_pretrained("microsoft/speecht5_hifigan")

speaker_embedding = torch.randn(1, 512)

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

SpeechT5ForTextToSpeech LOAD REPORT from: microsoft/speecht5_tts
Key                                         | Status     |  | 
--------------------------------------------+------------+--+-
speecht5.encoder.prenet.encode_positions.pe | UNEXPECTED |  | 
speecht5.decoder.prenet.encode_positions.pe | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/158 [00:00<?, ?it/s]

In [64]:
try:
    with open("unzipped_data/symptom-disease-dataset/mapping.json", "r") as f:
        label_mapping = json.load(f)
    reverse_label_mapping = {v: k for k, v in label_mapping.items()}
except Exception as e:
    print("Warning: Could not load mapping.join. Ensure it is formatted as JSON.")
    label_mapping = {}

In [89]:
import subprocess
import os
import soundfile as sf

def test_voice_consultation(audio_file_path):
    print(f"\n{'-'*40}")
    print(f"Processing Audio File: {audio_file_path}")
    print(f"{'-'*40}")

    # Convert webm to wav for better compatibility with ASR pipeline
    wav_audio_file_path = audio_file_path.replace('.webm', '.wav')
    try:
        subprocess.run(['ffmpeg', '-y', '-i', audio_file_path, '-ar', '16000', wav_audio_file_path], check=True, capture_output=True)
        print(f"   Converted {audio_file_path} to {wav_audio_file_path}")
    except subprocess.CalledProcessError as e:
        print(f"   Error converting audio file: {e.stderr.decode()}")
        return

    # Check if the converted WAV file exists and is not empty
    if not os.path.exists(wav_audio_file_path) or os.path.getsize(wav_audio_file_path) == 0:
        print(f"   Error: Converted WAV file '{wav_audio_file_path}' is empty or not created.")
        print("   Please ensure your microphone is working and you spoke clearly during recording.")
        return

    # Optionally, check audio properties with soundfile to ensure it's valid
    try:
        data, samplerate = sf.read(wav_audio_file_path)
        if data.size == 0:
            print(f"   Error: Converted WAV file '{wav_audio_file_path}' contains no audio data.")
            print("   Please ensure your microphone is working and you spoke clearly during recording.")
            return
        print(f"   WAV file properties: Sample Rate={samplerate}, Duration={len(data)/samplerate:.2f}s")
    except Exception as e:
        print(f"   Error reading WAV file with soundfile: {e}")
        print("   The converted audio file might be corrupted.")
        return

    print("1. Transcribing your speech...")
    transcription = stt_pipeline(wav_audio_file_path)["text"]
    print(f"   [You said]: {transcription}")

    print("\n 2. Running text through your 8-bit model...")

    inputs = tokenizer(
        transcription,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = quantized_model(**inputs)

    predicted_id = torch.argmax(outputs.logits, dim=-1).item()

    response_text = f"I hear you! Based on what you've described, it is likely to say that you have a problem of {reverse_label_mapping[predicted_id]}."
    print(f"   [AI Output Text]: {response_text}")

    print("\n 3. Generating AI voice response...")
    tts_inputs = tts_processor(text=response_text, return_tensors="pt")

    with torch.no_grad():
        speech_output = tts_model.generate_speech(
            tts_inputs["input_ids"],
            speaker_embedding,
            vocoder=vocoder
        )

    print("\n Done! Listen to the AI's response below:")
    display(Audio(speech_output.numpy(), rate=15500))


In [91]:
from IPython.display import Javascript, display
from google.colab import output
from base64 import b64decode

# JavaScript to access browser's microphone
RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
})
var record = async () => {
  stream = await navigator.mediaDevices.getUserMedia({audio: true})
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()

  // Create a clickable button to stop recording
  let div = document.createElement('div');
  div.innerHTML = '<b style="color:white; background-color:red; padding:10px; border-radius:5px; cursor:pointer;">🔴 Recording... Speak now! CLICK HERE TO STOP</b>';
  document.body.appendChild(div);

  return new Promise(resolve => {
    div.onclick = () => {
      recorder.stop()
      div.remove()
    }
    recorder.onstop = async () => {
      blob = new Blob(chunks)
      text = await b2text(blob)
      resolve(text)
    }
  })
}
"""

def record_microphone(filename='symptoms.webm'):
    print("Please grant microphone access to your browser if prompted.")
    display(Javascript(RECORD_JS))
    print("Microphone is listening...")

    audio_string = output.eval_js('record()')

    audio_bytes = b64decode(audio_string.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(audio_bytes)

    print(f"Audio successfully saved as '{filename}'!")
    return filename

audio_file = record_microphone("symptoms.webm")

Please grant microphone access to your browser if prompted.


<IPython.core.display.Javascript object>

Microphone is listening...
Audio successfully saved as 'symptoms.webm'!


In [92]:
test_voice_consultation("symptoms.webm")


----------------------------------------
Processing Audio File: symptoms.webm
----------------------------------------
   Converted symptoms.webm to symptoms.wav
   WAV file properties: Sample Rate=16000, Duration=10.98s
1. Transcribing your speech...
   [You said]:  having high fever and having vomiting and not feeling very

 2. Running text through your 8-bit model...
   [AI Output Text]: I hear you! Based on what you've described, it is likely to say that you have a problem of Bronchial Asthma.

 3. Generating AI voice response...

 Done! Listen to the AI's response below:
